In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dropout, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt

# We define 'today' to dynamically use the current date in file paths.
# For example, if today is 2025-07-03, it will be used to read or write sentiment files.
today = datetime.today().strftime('%Y-%m-%d')

# Here we create some necessary output directories if they do not already exist,
# such as "models" for saved models, "test_set_predictions" for output plots, etc.
for directory in ["models", "test_set_predictions", "training_loss", "metrics"]:
    os.makedirs(directory, exist_ok=True)

# This list contains our top 10 symbols (e.g., popular stock tickers).
top_10_symbols = [
    "AMZN",        # Amazon
    "AAPL",        # Apple
    "GOOGL",       # Alphabet (Class A)
    "005930.KS",   # Samsung Electronics (Korea Exchange)
    "2317.TW",     # Foxconn (Hon Hai Precision, Taiwan)
    "MSFT",        # Microsoft
    "JD",          # Jingdong (JD.com)
    "BABA",        # Alibaba (NYSE)
    "T",           # AT&T
    "META"         # Meta (Facebook)
]

# create_sequences is a helper function to generate input subsequences (X) and
# corresponding target values (y) for the LSTM. For example, if time_steps=60,
# each X is a 60-day window of data, and y is the 'Close' from the 61st day.
def create_sequences(data, time_steps=60):
    X, y = [], []
    for i in range(time_steps, len(data)):
        X.append(data[i - time_steps:i])
        y.append(data[i][0])  # 'Close' price is at index 0
    return np.array(X), np.array(y)

# inverse_close is used to reverse-scale predictions back to original price values.
# For example, if the LSTM predicts a scaled close of 0.8, we combine with zeros
# for the other scaled columns and use the scaler's inverse transform.
def inverse_close(preds, scaler):
    combined = np.concatenate([preds, np.zeros_like(preds)], axis=1)
    return scaler.inverse_transform(combined)[:, 0]

# print_metrics computes MAE, MSE, and R^2 score to evaluate prediction accuracy.
# For example, if true_vals=[10,12] and preds=[9,13], we can see the model's error.
def print_metrics(true_vals, preds):
    mae = mean_absolute_error(true_vals, preds)
    mse = mean_squared_error(true_vals, preds)
    r2 = r2_score(true_vals, preds)
    return mae, mse, r2

# We loop through each of the top_10_symbols, perform data loading, model training,
# plotting, and saving metrics. The try-except ensures one symbol failing won't
# break the entire run.
for symbol in top_10_symbols:
    try:
        # Load the CSV, skip its first row, reset index, and write it back.
        # This step might be for cleaning or removing an unwanted header row.
        df = pd.read_csv(f"../LSTM/datasets/{symbol}_daily_data.csv")
        df = df.iloc[1:].reset_index(drop=True)
        df.to_csv(f"../LSTM/datasets/{symbol}_daily_data.csv", index=False)

        print(f"\n=== Processing {symbol} ===")

        # === Load Data ===
        # Read the same CSV file again, parse dates for 'Date'.
        stock_df = pd.read_csv(f"../LSTM/datasets/{symbol}_daily_data.csv")
        stock_df['Date'] = pd.to_datetime(stock_df['Date'])

        # Read the sentiment file corresponding to the current date, parse dates,
        # and convert sentiment input into numeric score (POSITIVE=1, NEGATIVE=-1, NEUTRAL=0).
        sentiment_df = pd.read_csv(f"../Sentiment-Analysis/sentiment/{today}/{symbol}_sentiment.csv")
        sentiment_df['date'] = pd.to_datetime(sentiment_df['date'])
        sentiment_df['sentiment_score'] = sentiment_df['sentiment'].map({
            'POSITIVE': 1, 'NEGATIVE': -1, 'NEUTRAL': 0
        })

        # Compute average sentiment per day, rename columns, and merge with the stock data on 'Date'.
        daily_sentiment = sentiment_df.groupby('date')['sentiment_score'].mean().reset_index()
        daily_sentiment.columns = ['Date', 'Sentiment']
        merged_df = pd.merge(stock_df[['Date', 'Close']], daily_sentiment, on='Date', how='left')
        merged_df['Sentiment'].fillna(0, inplace=True)

        # === Scale Data ===
        # We use MinMaxScaler to transform the 'Close' and 'Sentiment' columns to [0,1].
        # This is generally recommended for neural networks that use activation functions
        # sensitive to input scale.
        scaler = MinMaxScaler()
        scaled_data = scaler.fit_transform(merged_df[['Close', 'Sentiment']])

        # Generate the X,y sequences using our helper. We define time_steps=60 days.
        time_steps = 60
        X, y = create_sequences(scaled_data, time_steps)

        # Split data into training and testing sets using an 80/20 ratio.
        split_index = int(len(X) * 0.8)
        X_train, y_train = X[:split_index], y[:split_index]
        X_test, y_test = X[split_index:], y[split_index:]

        # === Build Model ===
        # Sequential LSTM model with two LSTM layers and dropout to reduce overfitting,
        # then a Dense layer outputs the predicted Close (1 unit).
        model = Sequential([
            LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
            Dropout(0.1),
            LSTM(32),
            Dropout(0.1),
            Dense(1)
        ])
        model.compile(optimizer='adam', loss='mean_squared_error')

        # EarlyStopping monitors validation loss, stops training if no improvement for 10 epochs,
        # and ModelCheckpoint saves the best model.
        early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
        checkpoint_path = f"models/{symbol}_best_model.h5"
        checkpoint = ModelCheckpoint(checkpoint_path, save_best_only=True)

        # === Train Model ===
        # We train our model in a silent mode (verbose=0), with a default of up to 100 epochs.
        # The training tracks validation data for early stopping.
        history = model.fit(
            X_train, y_train,
            validation_data=(X_test, y_test),
            epochs=100,
            batch_size=32,
            callbacks=[early_stop, checkpoint],
            verbose=0
        )

        # === Predict ===
        # We generate predictions for training and testing sets.
        # For example, if we have 5000 training sequences, we get 5000 predictions.
        train_preds = model.predict(X_train)
        test_preds = model.predict(X_test)

        # Convert scaled predictions/targets back to real prices.
        train_prices = inverse_close(train_preds, scaler)
        test_prices = inverse_close(test_preds, scaler)
        true_train = inverse_close(y_train.reshape(-1, 1), scaler)
        true_test = inverse_close(y_test.reshape(-1, 1), scaler)

        # === Metrics ===
        # We now compare predictions versus actual prices with MAE, MSE, and R^2.
        train_mae, train_mse, train_r2 = print_metrics(true_train, train_prices)
        test_mae, test_mse, test_r2 = print_metrics(true_test, test_prices)

        # === Predict Next Day ===
        # We take the last 'time_steps' daily data to create a final sequence,
        # generate one-step-ahead forecast, then inverse-scale that to an actual price.
        last_sequence = scaled_data[-time_steps:].reshape(1, time_steps, 2)
        next_scaled_pred = model.predict(last_sequence)
        next_day_price = inverse_close(next_scaled_pred, scaler)[0]

        # === Save Metrics to CSV ===
        # We store the computed metrics and next_day_price in a CSV file
        # for potential additional analysis or logging.
        metrics_df = pd.DataFrame([{
            'symbol': symbol,
            'train_mae': round(train_mae, 4),
            'train_mse': round(train_mse, 4),
            'train_r2': round(train_r2, 4),
            'test_mae': round(test_mae, 4),
            'test_mse': round(test_mse, 4),
            'test_r2': round(test_r2, 4),
            'next_day_prediction': round(next_day_price, 4)
        }])
        metrics_df.to_csv(f"metrics/{symbol}_lstm_senti_model_metrics.csv", index=False)

        # === Plot Predictions ===
        # Plot actual vs predicted prices on the test set.
        # Save the plot to 'test_set_predictions' directory.
        plt.figure(figsize=(12, 6))
        plt.plot(true_test, label='Actual')
        plt.plot(test_prices, label='Predicted')
        plt.title(f"{symbol} - Stock Price Prediction (Test Set)")
        plt.xlabel("Time")
        plt.ylabel("Price")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"test_set_predictions/{symbol}_lstm_senti_test_plot.png")
        plt.close()

        # === Plot Loss ===
        # Plot the training and validation loss curves across epochs.
        # Save this figure to 'training_loss' directory.
        plt.figure(figsize=(8, 4))
        plt.plot(history.history['loss'], label='Training Loss')
        plt.plot(history.history['val_loss'], label='Validation Loss')
        plt.title(f"{symbol} - Loss Curve")
        plt.xlabel("Epochs")
        plt.ylabel("MSE Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(f"training_loss/{symbol}_lstm_senti_loss_plot.png")
        plt.close()

        print(f"✓ Completed {symbol}")

    except Exception as e:
        # In case of an error, we print a message but keep going for the remaining symbols.
        print(f"✗ Error processing {symbol}: {e}")


2025-07-14 11:49:45.241974: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-14 11:49:45.245687: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-14 11:49:45.254573: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752493785.268998   16613 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752493785.273248   16613 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752493785.285866   16613 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin


=== Processing AMZN ===


/tmp/ipykernel_16613/3147326109.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
2025-07-14 11:49:48.320549: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(sh

159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
✓ Completed AMZN

=== Processing AAPL ===


/tmp/ipykernel_16613/3147326109.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


...

/tmp/ipykernel_16613/3147326109.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


159/159 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step
✓ Completed 005930.KS

=== Processing 2317.TW ===


/tmp/ipykernel_16613/3147326109.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


158/158 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
✓ Completed 2317.TW

=== Processing MSFT ===


/tmp/ipykernel_16613/3147326109.py:90: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  merged_df['Sentiment'].fillna(0, inplace=True)
/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
